# Calculus and Optimization Mastery Lab

Implement stable multinomial logistic regression, verify its gradient, and compare optimization paths under a matched step budget. No autograd is used.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(14)
np.set_printoptions(precision=5, suppress=True)


## 1. Stable functions

Subtracting a row maximum changes neither softmax probabilities nor log-sum-exp differences, while preventing overflow.


In [ ]:
def logsumexp(x, axis=-1, keepdims=False):
    m = np.max(x, axis=axis, keepdims=True)
    out = m + np.log(np.sum(np.exp(x - m), axis=axis, keepdims=True))
    return out if keepdims else np.squeeze(out, axis=axis)

def softmax(logits):
    return np.exp(logits - logsumexp(logits, axis=1, keepdims=True))

extreme = np.array([[1000.0, 999.0, -1000.0], [-1000.0, -1001.0, -999.0]])
p = softmax(extreme)
assert np.all(np.isfinite(p))
assert np.allclose(p.sum(axis=1), 1.0)
print(p)


## 2. Synthetic three-class data

The split is fixed before any optimizer comparison. Standardization uses training statistics only.


In [ ]:
centers = np.array([[-2.0, 0.0], [2.0, 0.0], [0.0, 2.5]])
X = np.vstack([rng.normal(loc=c, scale=1.15, size=(200, 2)) for c in centers])
y = np.repeat(np.arange(3), 200)
perm = rng.permutation(len(X))
train, valid, test = perm[:360], perm[360:480], perm[480:]
mean, std = X[train].mean(axis=0), X[train].std(axis=0)
Xs = (X - mean) / std
X_train, y_train = Xs[train], y[train]
X_valid, y_valid = Xs[valid], y[valid]
X_test, y_test = Xs[test], y[test]
assert X_train.shape == (360, 2)


## 3. Loss and analytic gradients

For mean cross-entropy, the logit gradient is (p-y_one_hot)/N.


In [ ]:
def loss_and_grad(W, b, X, y):
    logits = X @ W + b
    log_probs = logits - logsumexp(logits, axis=1, keepdims=True)
    loss = -np.mean(log_probs[np.arange(len(X)), y])
    probs = np.exp(log_probs)
    probs[np.arange(len(X)), y] -= 1.0
    probs /= len(X)
    return loss, X.T @ probs, probs.sum(axis=0)

def accuracy(W, b, X, y):
    return np.mean(np.argmax(X @ W + b, axis=1) == y)

W0 = rng.normal(scale=0.1, size=(2, 3))
b0 = rng.normal(scale=0.1, size=3)
loss0, gW, gb = loss_and_grad(W0, b0, X_train[:17], y_train[:17])
assert gW.shape == W0.shape and gb.shape == b0.shape
print("initial small-batch loss:", loss0)


## 4. Central-difference gradient check

Test several step sizes: truncation error dominates when h is large; rounding and cancellation dominate when h is tiny.


In [ ]:
def numerical_gradient_W(W, b, X, y, h):
    out = np.zeros_like(W)
    for i in range(W.shape[0]):
        for j in range(W.shape[1]):
            plus, minus = W.copy(), W.copy()
            plus[i, j] += h
            minus[i, j] -= h
            out[i, j] = (loss_and_grad(plus, b, X, y)[0] - loss_and_grad(minus, b, X, y)[0]) / (2*h)
    return out

steps = np.logspace(-1, -8, 8)
errors = []
for h in steps:
    ng = numerical_gradient_W(W0, b0, X_train[:17], y_train[:17], h)
    errors.append(np.linalg.norm(ng-gW) / max(1e-12, np.linalg.norm(ng)+np.linalg.norm(gW)))
assert min(errors) < 1e-7
print(list(zip(steps, np.array(errors))))


## 5. Matched-budget optimizers

All methods receive the same 300 full-batch gradient evaluations. AdamW applies decoupled weight decay directly to W.


In [ ]:
def train(method, lr, steps=300, weight_decay=1e-3):
    W = np.zeros((2, 3)); b = np.zeros(3)
    vW = np.zeros_like(W); vb = np.zeros_like(b)
    mW = np.zeros_like(W); mb = np.zeros_like(b)
    sW = np.zeros_like(W); sb = np.zeros_like(b)
    history = []
    for t in range(1, steps+1):
        loss, gradW, gradb = loss_and_grad(W, b, X_train, y_train)
        if method == "gd":
            W -= lr * (gradW + weight_decay * W); b -= lr * gradb
        elif method == "momentum":
            vW = 0.9*vW + gradW + weight_decay*W; vb = 0.9*vb + gradb
            W -= lr*vW; b -= lr*vb
        elif method == "adamw":
            mW = 0.9*mW + 0.1*gradW; mb = 0.9*mb + 0.1*gradb
            sW = 0.999*sW + 0.001*gradW**2; sb = 0.999*sb + 0.001*gradb**2
            mhW, mhb = mW/(1-0.9**t), mb/(1-0.9**t)
            shW, shb = sW/(1-0.999**t), sb/(1-0.999**t)
            W *= (1 - lr*weight_decay)
            W -= lr*mhW/(np.sqrt(shW)+1e-8); b -= lr*mhb/(np.sqrt(shb)+1e-8)
        history.append((loss, accuracy(W,b,X_valid,y_valid), np.linalg.norm(gradW)))
    return W, b, np.array(history)

configs = {"gd":0.2, "momentum":0.08, "adamw":0.05}
results = {name: train(name, rate) for name, rate in configs.items()}
for name, (W,b,hist) in results.items():
    assert hist[-1,0] < hist[0,0]
    print(name, "valid", round(accuracy(W,b,X_valid,y_valid),3), "test", round(accuracy(W,b,X_test,y_test),3))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for name, (_,_,hist) in results.items():
    ax[0].plot(hist[:,0], label=name)
    ax[1].plot(hist[:,2], label=name)
ax[0].set(title="Training loss", xlabel="gradient evaluations")
ax[1].set(title="Gradient norm", xlabel="gradient evaluations", yscale="log")
for a in ax: a.legend()
plt.tight_layout()


## Exercises

1. Plot the gradient-check error against h on log-log axes and explain both sides of the curve.
2. Replace full-batch gradients with minibatches. Estimate gradient variance at a fixed parameter state.
3. Run a predeclared learning-rate grid and choose by validation loss only.
4. Add label noise and compare optimizer speed with test generalization.
5. Deliberately use naive exp(logits) on the extreme example, record the failure, and repair it.
6. Derive and verify one AdamW update by hand.

Carry the completed notebook into the Train and Audit a Multiclass Classifier capstone.
